# Entrenamiento y Evaluación
Fase 6: entrena GBM (y variantes) sobre el dataset integrado y reporta MAE/MAPE.

**Semilla fija:** `SEED = 42` — resultados reproducibles.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json, time
from pathlib import Path
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

sns.set_theme(style='whitegrid')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/tpp-ml-dataset')
except ImportError:
    BASE = Path('..').resolve()

PROCESSED = BASE / 'data/processed'
SEED = 42
MAPE_THRESHOLD = 25.0
print('BASE:', BASE)

## 1. Carga y split temporal

In [2]:
df = pd.read_csv(PROCESSED / 'events_with_features.csv', parse_dates=['date'])
df = df.sort_values(['product_id', 'date']).reset_index(drop=True)

# Split temporal: train hasta 2016, test 2017
cutoff = pd.Timestamp('2017-01-01')
train_df = df[df['date'] < cutoff].copy()
test_df  = df[df['date'] >= cutoff].copy()

print(f'Train: {len(train_df):,} filas  ({train_df["date"].min().date()} → {train_df["date"].max().date()})')
print(f'Test:  {len(test_df):,} filas   ({test_df["date"].min().date()} → {test_df["date"].max().date()})')

Train: 730,500 filas  (2013-01-01 → 2016-12-31)
Test:  182,500 filas   (2017-01-01 → 2017-12-31)


## 2. Preparación de matrices X / y

In [3]:
FEATURE_COLS = [
    'rolling_mean_7', 'rolling_mean_14', 'freq_per_week',
    'days_since_last', 'stock', 'stock_ratio',
    'dow', 'month', 'categoria'
]
TARGET = 'quantity'

# Eliminar filas con nulos en features (primer día de cada producto)
train_clean = train_df.dropna(subset=FEATURE_COLS)
test_clean  = test_df.dropna(subset=FEATURE_COLS)

X_train = train_clean[FEATURE_COLS].values
y_train = train_clean[TARGET].values
X_test  = test_clean[FEATURE_COLS].values
y_test  = test_clean[TARGET].values

print(f'X_train: {X_train.shape}   X_test: {X_test.shape}')

X_train: (730000, 9)   X_test: (182500, 9)


## 3. Funciones de métricas

In [4]:
def mape(y_true, y_pred):
    mask = y_true > 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    t0 = time.time()
    model.fit(X_tr, y_tr)
    elapsed = time.time() - t0
    preds = model.predict(X_te)
    mae  = mean_absolute_error(y_te, preds)
    mape_val = mape(y_te, preds)
    status = '✅ APROBADO' if mape_val <= MAPE_THRESHOLD else '❌ RECHAZADO'
    print(f'[{name}]  MAE={mae:.2f}  MAPE={mape_val:.2f}%  ({elapsed:.1f}s)  {status}')
    return {'model': name, 'mae': round(mae, 4), 'mape': round(mape_val, 4),
            'approved': mape_val <= MAPE_THRESHOLD, 'train_seconds': round(elapsed, 1)}

## 4. Entrenamiento — GradientBoosting (GBM)

In [5]:
results = []

gbm = GradientBoostingRegressor(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=SEED)
results.append(evaluate('GBM', gbm, X_train, y_train, X_test, y_test))

[GBM]  MAE=6.43  MAPE=13.22%  (57.2s)  ✅ APROBADO


## 5. Feature importances — GBM

In [ ]:
importances = pd.Series(gbm.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Feature Importances — GBM')
ax.set_xlabel('Importancia relativa')
plt.tight_layout()
plt.savefig(PROCESSED / 'feature_importances_gbm.png', dpi=120)
plt.show()
print(importances.sort_values(ascending=False).to_string())

## 6. Análisis de residuos

In [ ]:
preds_gbm = gbm.predict(X_test)
residuals = y_test - preds_gbm

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].scatter(preds_gbm[:2000], residuals[:2000], alpha=0.2, s=5, color='steelblue')
axes[0].axhline(0, color='red', linewidth=0.8)
axes[0].set_xlabel('Predicción')
axes[0].set_ylabel('Residuo')
axes[0].set_title('Residuos vs Predicción')

axes[1].hist(residuals, bins=60, color='steelblue', edgecolor='white')
axes[1].set_title('Distribución de residuos')

axes[2].scatter(y_test[:2000], preds_gbm[:2000], alpha=0.2, s=5, color='coral')
lim = max(y_test.max(), preds_gbm.max())
axes[2].plot([0, lim], [0, lim], 'r--', linewidth=0.8)
axes[2].set_xlabel('Real')
axes[2].set_ylabel('Predicho')
axes[2].set_title('Real vs Predicho')

plt.tight_layout()
plt.savefig(PROCESSED / 'residuos_gbm.png', dpi=120)
plt.show()

## 7. Predicción por producto (muestra)

In [ ]:
sample_pids = test_clean['product_id'].unique()[:3]
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

for ax, pid in zip(axes, sample_pids):
    mask = test_clean['product_id'] == pid
    sub = test_clean[mask].copy()
    sub['pred'] = gbm.predict(sub[FEATURE_COLS].values)
    ax.plot(sub['date'].values, sub['quantity'].values, label='Real', linewidth=0.8)
    ax.plot(sub['date'].values, sub['pred'].values, label='Predicho', linewidth=0.8, linestyle='--')
    ax.set_title(f'Producto {pid}')
    ax.legend(fontsize=7)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Real vs Predicho — Test 2017', fontsize=12)
plt.tight_layout()
plt.savefig(PROCESSED / 'prediccion_muestra.png', dpi=120)
plt.show()

## 8. Resumen final y exportación de resultados

In [9]:
results_df = pd.DataFrame(results)
print('='*55)
print('RESULTADOS FINALES')
print('='*55)
print(results_df.to_string(index=False))
print(f'\nUmbral MAPE del sistema: {MAPE_THRESHOLD}%')

# Convertir tipos numpy a tipos nativos de Python para JSON
def to_native(obj):
    if isinstance(obj, (np.integer,)): return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, (np.bool_,)): return bool(obj)
    return obj

results_serializable = [
    {k: to_native(v) for k, v in r.items()} for r in results
]

report = {
    'seed': SEED,
    'mape_threshold': MAPE_THRESHOLD,
    'train_rows': int(len(X_train)),
    'test_rows': int(len(X_test)),
    'train_period': {'from': str(train_df['date'].min().date()), 'to': str(train_df['date'].max().date())},
    'test_period': {'from': str(test_df['date'].min().date()), 'to': str(test_df['date'].max().date())},
    'features': FEATURE_COLS,
    'results': results_serializable,
    'feature_importances_gbm': {
        k: round(float(v), 6) for k, v in
        pd.Series(gbm.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False).items()
    }
}

with open(PROCESSED / 'evaluation_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print('\nevaluation_report.json guardado.')

RESULTADOS FINALES
model    mae    mape  approved  train_seconds
  GBM 6.4322 13.2173      True           57.2

Umbral MAPE del sistema: 25.0%

evaluation_report.json guardado.


## Conclusión

El pipeline GBM entrenado sobre el dataset integrado produce un MAPE evaluable contra el umbral del 25% del sistema.

- Dataset: 913k eventos, 500 productos, 2013–2017
- Features: rolling means, frecuencia, días desde consumo, stock, stock_ratio, calendario, categoría
- Split temporal: sin mezcla de fechas futuras
- Seed fija: resultados reproducibles

**La validación definitiva del sistema debe realizarse con datos reales generados por la aplicación.**